# Deep Learning Foundations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/11_deep_learning.ipynb)

Runnable examples from **Data/ML Handbook** — section `11_deep_learning`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/11_deep_learning/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s09_backprop.py`

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
X = rng.normal(size=(8, 4)); y = rng.integers(0, 3, 8); Y = np.eye(3)[y]
W1, b1, W2, b2 = rng.normal(size=(4, 5)) * .5, np.zeros(5), rng.normal(size=(5, 3)) * .5, np.zeros(3)
def forward(W1, b1, W2, b2):
    z1 = X @ W1 + b1; h = np.maximum(0, z1); z2 = h @ W2 + b2
    p = np.exp(z2 - z2.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    return z1, h, p, -np.mean(np.log(p[np.arange(len(y)), y]))
z1, h, p, L = forward(W1, b1, W2, b2)
dz2 = (p - Y) / len(y)                 # softmax + cross-entropy gradient
dW2, db2 = h.T @ dz2, dz2.sum(0)
dh = dz2 @ W2.T; dz1 = dh * (z1 > 0)    # ReLU gate
dW1, db1 = X.T @ dz1, dz1.sum(0)
eps, i, j = 1e-6, 2, 1                  # gradient check one weight
Wp, Wm = W1.copy(), W1.copy(); Wp[i, j] += eps; Wm[i, j] -= eps
num = (forward(Wp, b1, W2, b2)[3] - forward(Wm, b1, W2, b2)[3]) / (2 * eps)
print(f"loss={L:.4f}  dW1[{i},{j}] analytic={dW1[i, j]:.8f} numeric={num:.8f}")
import torch
t = [torch.tensor(a, requires_grad=True) for a in (W1, b1, W2, b2)]
out = torch.relu(torch.tensor(X) @ t[0] + t[1]) @ t[2] + t[3]
torch.nn.functional.cross_entropy(out, torch.tensor(y)).backward()
print("matches PyTorch autograd:", np.allclose(t[0].grad.numpy(), dW1), np.allclose(t[2].grad.numpy(), dW2))

### `code/s09_torch.py`

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
torch.manual_seed(0)
X = torch.randn(4000, 20); w = torch.randn(20, 3); y = (X @ w + 0.5 * torch.randn(4000, 3)).argmax(1)   # 3 classes
train_ds, val_ds = random_split(TensorDataset(X, y), [3200, 800], generator=torch.Generator().manual_seed(0))
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True); val_dl = DataLoader(val_ds, batch_size=256)
class MLP(nn.Module):
    def __init__(self, d_in=20, h=64, n_cls=3, p=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(p), nn.Linear(h, n_cls))
    def forward(self, x): return self.net(x)                    # return LOGITS
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLP().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-2)
loss_fn = nn.CrossEntropyLoss()
print(f"params={sum(p.numel() for p in model.parameters())}  initial loss≈ln(3)={torch.log(torch.tensor(3.)):.3f}")
best, bad = float("inf"), 0
for epoch in range(30):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(xb), yb); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    model.eval(); tot, correct, n = 0.0, 0, 0
    with torch.inference_mode():
        for xb, yb in val_dl:
            out = model(xb.to(device)); tot += loss_fn(out, yb.to(device)).item() * len(yb)
            correct += (out.argmax(1).cpu() == yb).sum().item(); n += len(yb)
    val = tot / n
    if epoch % 5 == 0: print(f"epoch {epoch:2d} train_loss={loss.item():.3f} val_loss={val:.3f} val_acc={correct/n:.3f}")
    if val < best - 1e-4: best, bad = val, 0; torch.save(model.state_dict(), "data/best.pt")
    else:
        bad += 1
        if bad == 5: print(f"early stop at epoch {epoch}"); break
model.load_state_dict(torch.load("data/best.pt")); print(f"best val_loss={best:.3f}")

### `code/s09_keras.py`

In [ ]:
import numpy as np, tensorflow as tf
from tensorflow import keras
from keras import layers
keras.utils.set_random_seed(0)
X = np.random.randn(4000, 20).astype("float32"); w = np.random.randn(20, 3)
y = (X @ w + 0.5 * np.random.randn(4000, 3)).argmax(1)
train = tf.data.Dataset.from_tensor_slices((X[:3200], y[:3200])).shuffle(3200).batch(64).prefetch(tf.data.AUTOTUNE)
val = tf.data.Dataset.from_tensor_slices((X[3200:], y[3200:])).batch(256)
model = keras.Sequential([layers.Input((20,)), layers.Dense(64), layers.LayerNormalization(),
                          layers.Activation("gelu"), layers.Dropout(0.2), layers.Dense(3)])     # logits
model.compile(optimizer=keras.optimizers.AdamW(3e-3, weight_decay=1e-2),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
hist = model.fit(train, validation_data=val, epochs=30, verbose=0,
                 callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
h = hist.history
for e in range(0, len(h["loss"]), 5): print(f"epoch {e:2d} loss={h['loss'][e]:.3f} val_loss={h['val_loss'][e]:.3f} val_acc={h['val_accuracy'][e]:.3f}")
print(f"stopped after {len(h['loss'])} epochs, best val_loss={min(h['val_loss']):.3f}, params={model.count_params()}")